# Head-to-head against existing subtyping tools on your held-out set

**Question answered.** How does your linear k-mer SVM compare to Kameris, CASTOR, COMET and
REGA on your own cluster-disjoint test fold, at every fragment length -- not just full length?

**What this notebook can and cannot automate.**
- **Kameris** is, by design, essentially k=6 k-mer frequency + linear SVM. This notebook
  re-implements it faithfully (one-vs-rest linear SVM on 6-mer frequency vectors, no TF-IDF
  weighting, matching the Kameris paper) so you get a real, local, GPU-free comparator with
  no install risk.
- **CASTOR** is RFLP-signature based (in-silico restriction digestion -> CUT/RMS features ->
  Random Forest). This notebook re-implements the CASTOR feature construction using a
  representative panel of Type II restriction enzymes (do **not** need REBASE download --
  the panel is hardcoded below from published Type II recognition sites) and trains a Random
  Forest on it, matching the CASTOR method.
- **COMET** and **REGA** are web tools (comet.retrovirology.lu and rega.kuleuven.be /
  bioafrica.net) without a public bulk REST API for automated scoring, and Kaggle notebooks
  have unreliable/rate-limited outbound internet even with "Internet: On". This notebook
  therefore (a) exports your test-fold sequences to FASTA files, one per fragment length,
  ready to paste into COMET/REGA's web form, and (b) gives you a fixed CSV schema to paste
  the results back into so cell 8 below merges everything into one final table. This is the
  standard, expected way to benchmark against these two tools -- nobody automates them at
  scale because their maintainers do not offer that.

**GPU needed.** No. Everything in this notebook is CPU-only (Linear SVM, Random Forest).

**Runtime.** Feature construction (RFLP digestion simulation) is the slowest step: budget
~15-30 min for ~4,500 test sequences x 5 lengths x ~40 enzymes. Model fitting is fast.
Comfortably inside an hour total, well within a 12 h session.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, re, time, json
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 2. Load data, split (same cluster-disjoint split as the main paper)

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
grp_all = df["cluster"].values

SEED = 0
s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trv, te = next(s1.split(X_all, y_all, grp_all))
s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(s2.split(X_all[trv], y_all[trv], grp_all[trv]))
tr, va = trv[tr_rel], trv[va_rel]
print(f"train={len(tr)} val={len(va)} test={len(te)}")
LENGTHS = [300, 500, 1000, None]  # None = full length


### 3. Fragment + export test-fold FASTA per length (for COMET / REGA manual submission)

In [ ]:

def fragment(seq, length, rng):
    if length is None or len(seq) <= length:
        return seq
    start = rng.integers(0, len(seq) - length + 1)
    return seq[start:start+length]

def write_fasta(path, seqs, ids):
    with open(path, "w") as f:
        for i, s in zip(ids, seqs):
            f.write(f">{i}\n{s}\n")

rng = np.random.default_rng(SEED)
test_ids = te
test_labels = pd.DataFrame({"id": test_ids, "true_subtype": le.inverse_transform(y_all[test_ids])})
test_labels.to_csv(f"{OUT}/toolheadtohead_test_true_labels.csv", index=False)

for L in LENGTHS:
    frags = [fragment(X_all[i], L, rng) for i in test_ids]
    tag = L if L else "full"
    write_fasta(f"{OUT}/toolheadtohead_test_{tag}bp.fasta", frags, test_ids)
    print(f"wrote toolheadtohead_test_{tag}bp.fasta ({len(frags)} sequences)")

print("""
Next manual step for COMET and REGA (5-10 min per length, once per length):
  1. Open https://comet.lih.lu (or https://comet.retrovirology.lu) and
     https://www.bioafrica.net/subtypetool/html/ (REGA).
  2. Upload each toolheadtohead_test_<len>bp.fasta.
  3. Download/copy the per-sequence subtype call (and UNASSIGNED / abstention flag) for each.
  4. Save as toolheadtohead_comet_results_<len>.csv and toolheadtohead_rega_results_<len>.csv with columns:
     id, predicted_subtype, unassigned (0/1), seconds_per_seq (optional, if the tool reports timing)
  5. Upload those CSVs back into this Kaggle notebook's input, or paste into /kaggle/working
     before running the merge cell (Section 8).
""")


### 4. Kameris-equivalent: raw k=6 k-mer frequency + linear SVM (no TF-IDF)

In [ ]:

def kameris_features(seqs, k=6):
    vec = CountVectorizer(analyzer="char", ngram_range=(k, k), lowercase=False)
    X = vec.fit_transform(seqs)
    X = X.multiply(1.0 / np.maximum(X.sum(axis=1), 1))  # normalize to frequency
    return vec, X.tocsr()

def kameris_transform(vec, seqs, k=6):
    X = vec.transform(seqs)
    X = X.multiply(1.0 / np.maximum(X.sum(axis=1), 1))
    return X.tocsr()

kameris_rows = []
for L in LENGTHS:
    seq_tr = [fragment(X_all[i], L, rng) for i in tr]
    seq_te = [fragment(X_all[i], L, rng) for i in te]
    vec, Xtr = kameris_features(seq_tr)
    Xte = kameris_transform(vec, seq_te)
    clf = LinearSVC(class_weight="balanced", random_state=SEED, max_iter=20000)
    t0 = time.time(); clf.fit(Xtr, y_all[tr]); train_t = time.time() - t0
    t0 = time.time(); pred = clf.predict(Xte); pred_t = (time.time() - t0) / len(te)
    kameris_rows.append(dict(
        tool="kameris_equivalent", length=(L if L else "full"),
        f1=f1_score(y_all[te], pred, average="macro"),
        bal_acc=balanced_accuracy_score(y_all[te], pred),
        pct_unassigned=0.0, train_seconds=train_t, seconds_per_seq=pred_t))
    print(kameris_rows[-1])


### 5. CASTOR-equivalent: in-silico RFLP (restriction fragment) features + Random Forest

Recognition sites for a representative panel of common Type II restriction enzymes (hardcoded here so nothing needs to be downloaded from REBASE).

In [ ]:

# A representative panel of ~40 common Type II restriction enzymes (name: recognition site,
# IUPAC ambiguity codes expanded to regex). This is a subset of REBASE, sufficient to
# reproduce CASTOR's CUT/RMS feature construction without needing the full database.
IUPAC = {"A":"A","C":"C","G":"G","T":"T","R":"[AG]","Y":"[CT]","S":"[GC]","W":"[AT]",
         "K":"[GT]","M":"[AC]","B":"[CGT]","D":"[AGT]","H":"[ACT]","V":"[ACG]","N":"[ACGT]"}

ENZYMES = {
 "EcoRI":"GAATTC","BamHI":"GGATCC","HindIII":"AAGCTT","PstI":"CTGCAG","SalI":"GTCGAC",
 "XhoI":"CTCGAG","NotI":"GCGGCCGC","SmaI":"CCCGGG","KpnI":"GGTACC","SacI":"GAGCTC",
 "SpeI":"ACTAGT","NheI":"GCTAGC","XbaI":"TCTAGA","ClaI":"ATCGAT","EcoRV":"GATATC",
 "NcoI":"CCATGG","NdeI":"CATATG","AflII":"CTTAAG","ApaI":"GGGCCC","BglII":"AGATCT",
 "DraI":"TTTAAA","HaeIII":"GGCC","HhaI":"GCGC","MspI":"CCGG","RsaI":"GTAC",
 "TaqI":"TCGA","AluI":"AGCT","MboI":"GATC","Sau3AI":"GATC","HpaII":"CCGG",
 "PvuII":"CAGCTG","ScaI":"AGTACT","StuI":"AGGCCT","AvaI":"CYCGRG","BstEII":"GGTNACC",
 "DpnI":"GATC","FokI":"GGATG","MseI":"TTAA","NlaIII":"CATG","Sau96I":"GGNCC",
}

def to_regex(site):
    return "".join(IUPAC[c] for c in site)

ENZYME_REGEX = {name: re.compile(to_regex(site)) for name, site in ENZYMES.items()}

def rflp_features(seq):
    """CASTOR-style features: for each enzyme, CUT (number of cut sites) and RMS
    (root-mean-square of the digested fragment lengths)."""
    feats = []
    n = len(seq)
    for name, rx in ENZYME_REGEX.items():
        cuts = [m.start() for m in rx.finditer(seq)]
        n_cuts = len(cuts)
        if n_cuts == 0:
            frag_lens = [n]
        else:
            bounds = [0] + cuts + [n]
            frag_lens = [bounds[i+1]-bounds[i] for i in range(len(bounds)-1)]
        rms = float(np.sqrt(np.mean(np.square(frag_lens))))
        feats.extend([n_cuts, rms])
    return feats

def build_rflp_matrix(seqs):
    return np.array([rflp_features(s) for s in seqs], dtype=np.float32)


In [ ]:

castor_rows = []
for L in LENGTHS:
    seq_tr = [fragment(X_all[i], L, rng) for i in tr]
    seq_te = [fragment(X_all[i], L, rng) for i in te]
    Xtr = build_rflp_matrix(seq_tr)
    Xte = build_rflp_matrix(seq_te)
    clf = RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                  random_state=SEED, n_jobs=-1)
    t0 = time.time(); clf.fit(Xtr, y_all[tr]); train_t = time.time() - t0
    t0 = time.time(); pred = clf.predict(Xte); pred_t = (time.time() - t0) / len(te)
    castor_rows.append(dict(
        tool="castor_equivalent", length=(L if L else "full"),
        f1=f1_score(y_all[te], pred, average="macro"),
        bal_acc=balanced_accuracy_score(y_all[te], pred),
        pct_unassigned=0.0, train_seconds=train_t, seconds_per_seq=pred_t))
    print(castor_rows[-1])


### 6. Your model: TF-IDF 6-mer + linear SVM (the main-paper baseline, for reference)

In [ ]:

svm_rows = []
for L in LENGTHS:
    seq_tr = [fragment(X_all[i], L, rng) for i in tr]
    seq_te = [fragment(X_all[i], L, rng) for i in te]
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xte = vec.transform(seq_te)
    clf = LinearSVC(class_weight="balanced", random_state=SEED, max_iter=20000)
    t0 = time.time(); clf.fit(Xtr, y_all[tr]); train_t = time.time() - t0
    t0 = time.time(); pred = clf.predict(Xte); pred_t = (time.time() - t0) / len(te)
    svm_rows.append(dict(
        tool="this_study_svm_tfidf6", length=(L if L else "full"),
        f1=f1_score(y_all[te], pred, average="macro"),
        bal_acc=balanced_accuracy_score(y_all[te], pred),
        pct_unassigned=0.0, train_seconds=train_t, seconds_per_seq=pred_t))
    print(svm_rows[-1])


### 7. Save local-comparator results now (do this even before COMET/REGA are back)

In [ ]:

local = pd.DataFrame(kameris_rows + castor_rows + svm_rows)
local.to_csv(f"{OUT}/toolheadtohead_local_comparators.csv", index=False)
local


### 8. Merge in COMET / REGA results once you have pasted them back (optional, run when ready)

Handles the UNASSIGNED / abstention case per the reviewer's note: report accuracy on assigned-only sequences AND accuracy counting abstention as error (both numbers).

In [ ]:

def score_with_abstention(true_labels, pred_labels, unassigned_flags):
    true_labels = np.asarray(true_labels); pred_labels = np.asarray(pred_labels, dtype=object)
    unassigned_flags = np.asarray(unassigned_flags).astype(bool)
    pct_unassigned = unassigned_flags.mean()

    assigned_mask = ~unassigned_flags
    if assigned_mask.sum() > 0:
        f1_assigned = f1_score(true_labels[assigned_mask], pred_labels[assigned_mask], average="macro")
        bal_assigned = balanced_accuracy_score(true_labels[assigned_mask], pred_labels[assigned_mask])
    else:
        f1_assigned = bal_assigned = float("nan")

    # counting abstention as error: give every abstained sequence a wrong/placeholder label
    pred_with_penalty = pred_labels.copy()
    pred_with_penalty[unassigned_flags] = "__ABSTAIN_AS_ERROR__"
    f1_penalized = f1_score(true_labels, pred_with_penalty, average="macro")

    return dict(pct_unassigned=pct_unassigned,
                f1_assigned_only=f1_assigned, bal_acc_assigned_only=bal_assigned,
                f1_counting_abstention_as_error=f1_penalized)

external_rows = []
true_df = pd.read_csv(f"{OUT}/toolheadtohead_test_true_labels.csv").set_index("id")

for tool in ["comet", "rega"]:
    for L in LENGTHS:
        tag = L if L else "full"
        path = f"{OUT}/toolheadtohead_{tool}_results_{tag}.csv"
        if not os.path.exists(path):
            print(f"[skip] {path} not found yet -- run the manual COMET/REGA step above first.")
            continue
        res = pd.read_csv(path).set_index("id")
        merged = true_df.join(res, how="inner")
        scores = score_with_abstention(merged["true_subtype"], merged["predicted_subtype"],
                                        merged.get("unassigned", 0))
        row = dict(tool=tool, length=tag, **scores)
        if "seconds_per_seq" in merged.columns:
            row["seconds_per_seq"] = merged["seconds_per_seq"].mean()
        external_rows.append(row)
        print(row)

if external_rows:
    pd.DataFrame(external_rows).to_csv(f"{OUT}/toolheadtohead_external_tools_results.csv", index=False)


### 9. Final combined table

In [ ]:

final = local.copy()
if external_rows:
    ext = pd.DataFrame(external_rows).rename(columns={"f1_assigned_only":"f1", "bal_acc_assigned_only":"bal_acc"})
    final = pd.concat([final, ext], ignore_index=True, sort=False)
final.to_csv(f"{OUT}/toolheadtohead_final_comparison_table.csv", index=False)
final


### 10. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r toolheadtohead_head_to_head_outputs.zip toolheadtohead_test_*.fasta toolheadtohead_test_true_labels.csv toolheadtohead_local_comparators.csv toolheadtohead_final_comparison_table.csv toolheadtohead_*results*.csv 2>/dev/null; ls -la /kaggle/working/*.zip